# ComfyUI + FLUX.3 Image Klein (INT8 ConvRot) — Google Colab 一键部署

本笔记完整复现当前 Colab 实例配置，包括：
- **ComfyUI**（最新版，含 T4/低算力 CUDA BF16 兼容补丁）
- **模型仓库**：`black-forest-labs/flux-3-image-klein-comfy`
  - Diffusion：`flux3-image-klein-int8-convrot.safetensors`（5.5 GB）
  - Text Encoder：`qwen3vl_4b_int8_convrot.safetensors`（4.6 GB）
  - Prompt Upsampler（可选）：`qwen3.5_9b_flux3_prompt_upsampler_w4a8.safetensors`（6.3 GB）
  - VAE：`flux2-vae.safetensors`（321 MB）
- **官方 INT8 工作流**：`flux3_klein_int8_t2i.json` / `i2i.json` / `mi2i.json` 及 5 张示例参考图
- **公网访问**：`cloudflared` 临时隧道映射 `http://127.0.0.1:8188`（默认 `--protocol http2` 走 TCP，避免 Colab 容器 UDP/QUIC 缓冲受限导致的极低吞吐）

### 显卡选择建议（重要）
- **推荐：L4（24GB VRAM, `sm_89`）或 A100（`sm_80`）** — 完整支持 `comfy_kitchen` 的 **W4A8 CUDA 融合核**、**GatedDeltaNet 解码核**与原生 **BF16**，Prompt Upsampler（Qwen3.5-9B W4A8）可全速运行。
- **兼容：T4（15GB VRAM, `sm_75`）** — 主模型（`int8-convrot`）生图很快，但 W4A8 算子要求 `sm_80+`，在 T4 上开启 Prompt Upsampler 会回退到纯 Eager 软解（约 `2.5s/token`）。如使用 T4，建议在工作流中将 **`Upsampler on/off` 设为 `false`**。

In [ ]:
#@title 1. 检查 GPU、算力架构与系统资源
import os
import shutil
import subprocess
import torch

assert torch.cuda.is_available(), "未检测到 GPU！请在 Colab 菜单栏：代码执行程序 -> 更改运行时类型 -> 选择 L4 / A100 / T4 GPU。"

gpu_name = torch.cuda.get_device_name(0)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024 ** 3)
disk_free_gb = shutil.disk_usage("/content").free / (1024 ** 3)

print(f"GPU 型号       : {gpu_name}")
print(f"Compute Cap    : sm_{cc_major}{cc_minor} ({cc_major}.{cc_minor})")
print(f"显存 (VRAM)    : {vram_gb:.1f} GB")
print(f"系统内存 (RAM) : {ram_gb:.1f} GB")
print(f"磁盘可用 (/content): {disk_free_gb:.1f} GB")
print(f"PyTorch 版本   : {torch.__version__}")

if (cc_major, cc_minor) >= (8, 0):
    print("\n[OK] 当前显卡满足 sm_80+，W4A8 Prompt Upsampler、GatedDeltaNet 与原生 BF16 均可硬件加速。")
else:
    print(
        "\n[提示] 当前显卡为 sm_" + f"{cc_major}{cc_minor} (< sm_80)：\n"
        "  - FLUX.3 Klein INT8 生图本身可正常硬件加速；\n"
        "  - 但 Qwen3.5-9B W4A8 Prompt Upsampler 仅在 sm_80+（L4/A100 等）有 CUDA 融合核，"
        "在当前显卡上会回退到慢速 Eager 反量化（约 2.5s/token）。\n"
        "  - 建议切换为 L4/A100，或在工作流中将 Upsampler on/off 设为 false。"
    )

print()
print(subprocess.run(["nvidia-smi"], text=True, capture_output=True).stdout)

In [ ]:
#@title 2. 安装 ComfyUI、依赖与 cloudflared
COMFYUI_DIR = "/content/ComfyUI"  #@param {type:"string"}
PIN_COMMIT = ""  #@param {type:"string"}  # 留空拉取最新 master；如需严格锁定当前实例版本可填 fa68736

import os
import subprocess

def run_cmd(cmd, cwd=None):
    print(f">>> {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

# 1) 克隆或更新 ComfyUI
if not os.path.isdir(COMFYUI_DIR):
    run_cmd(f"git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_DIR}")
else:
    print(f"ComfyUI 目录已存在：{COMFYUI_DIR}")

if PIN_COMMIT.strip():
    run_cmd(f"git fetch --all && git checkout {PIN_COMMIT.strip()}", cwd=COMFYUI_DIR)

# 2) 安装 ComfyUI Python 依赖及 HuggingFace 高速下载组件
run_cmd("pip install -q -r requirements.txt", cwd=COMFYUI_DIR)
run_cmd("pip install -q -U huggingface_hub hf_xet hf_transfer")

# 3) 安装 cloudflared（用于无账号暴露本地 8188 端口）
if not os.path.isfile("/usr/local/bin/cloudflared"):
    run_cmd(
        "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"
    )
run_cmd("cloudflared --version")

print("\n[完成] ComfyUI 与 cloudflared 安装完毕。")

In [ ]:
#@title 3. 应用当前实例的 CUDA BF16 兼容补丁（主要针对 T4 / sm_75，L4/A100 下无副作用）
APPLY_CUDA_BF16_PATCH = True  #@param {type:"boolean"}
COMFYUI_DIR = "/content/ComfyUI"  #@param {type:"string"}

import os

PATCHES = [
    (
        os.path.join(COMFYUI_DIR, "comfy/text_encoders/llama.py"),
        "return torch.bfloat16 if comfy.model_management.should_use_bf16(device) else torch.float32",
        "return torch.bfloat16 if (comfy.model_management.should_use_bf16(device) or torch.device(device).type == 'cuda') else torch.float32",
    ),
    (
        os.path.join(COMFYUI_DIR, "comfy/text_encoders/qwen35.py"),
        "dt = torch.bfloat16 if comfy.model_management.should_use_bf16(device) else torch.float32",
        "dt = torch.bfloat16 if (comfy.model_management.should_use_bf16(device) or torch.device(device).type == 'cuda') else torch.float32",
    ),
]

if not APPLY_CUDA_BF16_PATCH:
    print("已跳过补丁。")
else:
    for filepath, old_str, new_str in PATCHES:
        if not os.path.isfile(filepath):
            print(f"[跳过] 文件不存在：{filepath}")
            continue
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        if new_str in content:
            print(f"[已存在] 补丁已应用：{filepath}")
        elif old_str in content:
            content = content.replace(old_str, new_str, 1)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"[成功] 已应用补丁：{filepath}")
        else:
            print(f"[警告] 未找到目标行（上游代码可能已更新）：{filepath}")

In [ ]:
#@title 4. 下载 FLUX.3 Klein INT8 模型、官方工作流与示例参考图
REPO_ID = "black-forest-labs/flux-3-image-klein-comfy"  #@param {type:"string"}
HF_TOKEN_INPUT = ""  #@param {type:"string"}  # 留空则自动读取环境变量 HF_TOKEN 或 Colab Secrets 中的 HF_TOKEN
COMFYUI_DIR = "/content/ComfyUI"  #@param {type:"string"}
DOWNLOAD_UPSAMPLER = True  #@param {type:"boolean"}  # 6.3GB；若在 T4 上不使用 Prompt Upsampler 可取消勾选
USE_DRIVE_MODEL_CACHE = False  #@param {type:"boolean"}  # 若勾选则把模型缓存到 Google Drive 以免下次重复下载
DRIVE_MODELS_DIR = "/content/drive/MyDrive/ComfyUI_models/flux3_klein_int8"  #@param {type:"string"}

import os
import shutil

# 启用 HuggingFace Xet 高速下载
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# 读取 HF_TOKEN（该仓库为 Gated Repo，需先在 HuggingFace 页面同意许可协议并提供 HF_TOKEN）
hf_token = HF_TOKEN_INPUT.strip() or os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    raise RuntimeError(
        f"{REPO_ID} 是受限仓库（Gated Repo），需要 HuggingFace Token 才能下载模型。\n"
        "请在 Colab 左侧【密钥 / Secrets】添加名为 HF_TOKEN 的密钥（并开启笔记本访问权限），"
        "或在上方 HF_TOKEN_INPUT 填入你的 Token（确保已在 HuggingFace 仓库页同意使用协议）。"
    )

from huggingface_hub import hf_hub_download

models_dir = os.path.join(COMFYUI_DIR, "models")
if USE_DRIVE_MODEL_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    target_models_dir = DRIVE_MODELS_DIR
else:
    target_models_dir = models_dir

MODEL_FILES = [
    "vae/flux2-vae.safetensors",
    "text_encoders/qwen3vl_4b_int8_convrot.safetensors",
    "diffusion_models/flux3-image-klein-int8-convrot.safetensors",
]
if DOWNLOAD_UPSAMPLER:
    MODEL_FILES.append("text_encoders/qwen3.5_9b_flux3_prompt_upsampler_w4a8.safetensors")

WORKFLOW_FILES = [
    "workflows/flux3_klein_int8_t2i.json",
    "workflows/flux3_klein_int8_i2i.json",
    "workflows/flux3_klein_int8_mi2i.json",
]

REF_IMAGE_FILES = [
    "workflows/klein_ref_autumn_forest.jpg",
    "workflows/klein_ref_great_wave.jpg",
    "workflows/klein_ref_green_doors.jpg",
    "workflows/klein_ref_toast.jpg",
    "workflows/klein_ref_woman_hat.jpg",
]

print("=== 1) 下载模型文件 ===")
for rel_path in MODEL_FILES:
    print(f"下载 {rel_path} ...")
    local_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=rel_path,
        local_dir=target_models_dir,
        token=hf_token,
    )
    # 如果启用了 Google Drive 缓存，将模型软链接回 /content/ComfyUI/models 对应子目录
    if USE_DRIVE_MODEL_CACHE:
        link_path = os.path.join(models_dir, rel_path)
        os.makedirs(os.path.dirname(link_path), exist_ok=True)
        if os.path.islink(link_path) or os.path.exists(link_path):
            os.remove(link_path)
        os.symlink(local_path, link_path)
    size_mb = os.path.getsize(local_path) / (1024 ** 2)
    print(f"  -> 就绪: {local_path} ({size_mb:.1f} MB)")

print("\n=== 2) 下载官方 INT8 工作流 ===")
wf_dir = os.path.join(COMFYUI_DIR, "user/default/workflows")
os.makedirs(wf_dir, exist_ok=True)
for rel_path in WORKFLOW_FILES:
    cached = hf_hub_download(repo_id=REPO_ID, filename=rel_path, token=hf_token)
    dst = os.path.join(wf_dir, os.path.basename(rel_path))
    shutil.copy2(cached, dst)
    print(f"  -> {dst}")

print("\n=== 3) 下载示例参考图到 input/ ===")
input_dir = os.path.join(COMFYUI_DIR, "input")
os.makedirs(input_dir, exist_ok=True)
for rel_path in REF_IMAGE_FILES:
    cached = hf_hub_download(repo_id=REPO_ID, filename=rel_path, token=hf_token)
    dst = os.path.join(input_dir, os.path.basename(rel_path))
    shutil.copy2(cached, dst)
    print(f"  -> {dst}")

print("\n[完成] 所有模型、工作流与参考图已准备完毕。")

In [ ]:
#@title 5. 启动 ComfyUI 与 Cloudflared 公网隧道
COMFYUI_DIR = "/content/ComfyUI"  #@param {type:"string"}
PORT = 8188  #@param {type:"integer"}
TUNNEL_PROTOCOL = "http2"  #@param ["http2", "quic", "auto"]
LOW_RAM_FLAGS = "auto"  #@param ["auto", "on", "off"]
# auto: 系统内存 < 20GB（如 T4 Standard 12GB）时自动启用 --disable-smart-memory --cache-none，与当前实例完全一致；
#       L4 / A100 高内存实例则保留智能显存/缓存以加速连续生图。
# on  : 强制启用 --disable-smart-memory --cache-none（严格复现当前 T4 启动参数）。
# off : 不附加低内存参数。

import os
import re
import subprocess
import time
import urllib.request

COMFY_LOG = "/content/comfyui.log"
CF_LOG = "/content/cloudflared.log"

# 1) 清理可能残留的旧进程（允许重复运行本单元格）
subprocess.run(["pkill", "-f", f"{COMFYUI_DIR}/main.py"], check=False)
subprocess.run(["pkill", "-f", f"cloudflared tunnel.*127.0.0.1:{PORT}"], check=False)
time.sleep(1)

# 2) 组装启动参数
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024 ** 3)
use_low_ram = (LOW_RAM_FLAGS == "on") or (LOW_RAM_FLAGS == "auto" and ram_gb < 20.0)

cmd = [
    "python3",
    os.path.join(COMFYUI_DIR, "main.py"),
    "--listen", "127.0.0.1",
    "--port", str(PORT),
]
if use_low_ram:
    cmd += ["--disable-smart-memory", "--cache-none"]

env = os.environ.copy()
env["HF_XET_HIGH_PERFORMANCE"] = "1"

print(f"启动 ComfyUI (RAM={ram_gb:.1f} GB, low_ram_flags={use_low_ram}):")
print(" ", " ".join(cmd))

comfy_log_f = open(COMFY_LOG, "w", encoding="utf-8")
comfy_proc = subprocess.Popen(
    cmd,
    cwd=COMFYUI_DIR,
    stdout=comfy_log_f,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)

# 3) 等待 ComfyUI 就绪
ready = False
for _ in range(90):
    if comfy_proc.poll() is not None:
        raise RuntimeError(f"ComfyUI 启动失败（exit={comfy_proc.returncode}），请查看 {COMFY_LOG}")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/system_stats", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    raise TimeoutError(f"等待 ComfyUI 端口 {PORT} 超时，请查看 {COMFY_LOG}")
print(f"[OK] ComfyUI 已启动 (PID={comfy_proc.pid})")

# 4) 启动 cloudflared 隧道并提取公网地址（默认 --protocol http2 走 TCP，避开 Colab 容器 UDP/QUIC 缓冲受限问题）
cf_cmd = ["cloudflared", "tunnel", "--no-autoupdate"]
if TUNNEL_PROTOCOL != "auto":
    cf_cmd += ["--protocol", TUNNEL_PROTOCOL]
cf_cmd += ["--url", f"http://127.0.0.1:{PORT}"]
print("启动 cloudflared:", " ".join(cf_cmd))

cf_log_f = open(CF_LOG, "w", encoding="utf-8")
cf_proc = subprocess.Popen(
    cf_cmd,
    stdout=cf_log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

tunnel_url = None
for _ in range(30):
    if cf_proc.poll() is not None:
        raise RuntimeError(f"cloudflared 启动失败（exit={cf_proc.returncode}），请查看 {CF_LOG}")
    if os.path.isfile(CF_LOG):
        text = open(CF_LOG, "r", errors="replace").read()
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", text)
        if m and "Registered tunnel connection" in text:
            tunnel_url = m.group(0)
            break
    time.sleep(1)

if not tunnel_url:
    raise TimeoutError(f"未在 {CF_LOG} 中检测到 trycloudflare 链接")

print("\n" + "=" * 68)
print(f"ComfyUI 公网访问地址: {tunnel_url}")
print("打开后可在左侧 Workflows 面板加载：")
print("  - flux3_klein_int8_t2i.json  (文生图)")
print("  - flux3_klein_int8_i2i.json  (单图编辑)")
print("  - flux3_klein_int8_mi2i.json (多图参考)")
print("=" * 68)

In [ ]:
#@title 6. 查看运行状态、公网地址、队列与最新日志
PORT = 8188  #@param {type:"integer"}
TAIL_LINES = 25  #@param {type:"integer"}

import json
import re
import subprocess
import urllib.request

print("=== GPU / 内存 ===")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw", "--format=csv"], text=True, capture_output=True).stdout.strip())
print(subprocess.run(["free", "-h"], text=True, capture_output=True).stdout.strip())

print("\n=== 公网隧道 ===")
try:
    cf_text = open("/content/cloudflared.log", errors="replace").read()
    urls = sorted(set(re.findall(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", cf_text)))
    print("URL:", urls[-1] if urls else "未找到")
except FileNotFoundError:
    print("未找到 /content/cloudflared.log")

print("\n=== ComfyUI 队列 ===")
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/queue", timeout=3) as r:
        q = json.loads(r.read().decode())
        print(f"运行中: {len(q.get('queue_running', []))} | 等待中: {len(q.get('queue_pending', []))}")
except Exception as e:
    print("无法连接 ComfyUI:", e)

print(f"\n=== /content/comfyui.log 最后 {TAIL_LINES} 行 ===")
try:
    with open("/content/comfyui.log", "rb") as f:
        tail = f.read()[-8192:].decode("utf-8", errors="replace").splitlines()[-TAIL_LINES:]
    print("\n".join(tail))
except FileNotFoundError:
    print("未找到 /content/comfyui.log")

In [ ]:
#@title 7. 常用运维操作（中断当前任务 / 打包下载输出图片 / 停止服务）
ACTION = "none"  #@param ["none", "interrupt_current_task", "zip_and_download_outputs", "stop_services"]
PORT = 8188  #@param {type:"integer"}

import os
import shutil
import subprocess
import urllib.request

if ACTION == "none":
    print("请选择 ACTION 后再运行本单元格。")
elif ACTION == "interrupt_current_task":
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/interrupt", data=b"", method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        print("已发送中断请求，状态码:", r.status)
elif ACTION == "zip_and_download_outputs":
    out_dir = "/content/ComfyUI/output"
    zip_base = "/content/comfyui_outputs"
    zip_path = shutil.make_archive(zip_base, "zip", out_dir)
    print(f"已打包: {zip_path} ({os.path.getsize(zip_path) / (1024 ** 2):.2f} MB)")
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        print("非 Colab 浏览器交互环境，请直接下载:", zip_path)
elif ACTION == "stop_services":
    subprocess.run(["pkill", "-f", "/content/ComfyUI/main.py"], check=False)
    subprocess.run(["pkill", "-f", f"cloudflared tunnel.*127.0.0.1:{PORT}"], check=False)
    print("已停止 ComfyUI 与 cloudflared 服务。")